# Large-Scale EV Smart Charging: A Four-Pillar AI Project
## IEOR E4010 — Artificial Intelligence for OR and Financial Engineering
### Columbia University · Spring 2026 · Final Project

---

### Project Overview

You are the fleet operations manager at a delivery company depot. Every evening, **20 electric delivery vans** return from their routes with partially depleted batteries. They must all be charged to **at least 90% SoC (State of Charge)** by their individual morning departure times. However, the depot’s shared **150 kW electrical transformer** cannot supply all 20 EVs at full power simultaneously (20 EVs x 11 kW = 220 kW demand vs. 150 kW supply). Your objective is to **minimize total electricity cost** while meeting every vehicle’s departure deadline and respecting the transformer capacity constraint.

This is a **constrained optimization problem** with coupling constraints (shared transformer), stochastic elements (varying arrival times, SoC levels, electricity prices), and a natural sequential decision structure — making it an ideal testbed for comparing AI approaches.

### What You Will Build

| Pillar | Module | What It Does | Key Idea |
|--------|--------|-------------|----------|
| **Machine Learning** | `ml_forecaster.py` | XGBoost forecasts next-hour electricity prices from engineered tabular features | Hand-crafted features + gradient boosting |
| **Deep Learning** | `dl_forecaster.py` | LSTM learns price patterns from raw sequential data | Automatic representation learning |
| **Reinforcement Learning** | `rl_agent.py` | PPO agent learns a real-time charging policy through simulation | Trial-and-error with no future knowledge |
| **Agentic AI** | `agentic_ai.py` | GPT-4o orchestrator with tool-calling ties everything together | LLM as decision coordinator |

### How This Notebook Works

- **All student work happens in this notebook** (`main.ipynb`). The `.py` files are **provided infrastructure** — they define the environment, data generation, baseline models, and evaluation utilities. **Do not modify the `.py` files.**
- Sections marked **TODO** are where you implement and experiment. All code runs as-is with baseline implementations; your job is to improve upon them.
- Run cells **top-to-bottom**. Each section builds on the previous one.

## 0. Setup & Configuration

Run the cell below to install all dependencies. If you are on Colab, it will also clone the repository.

In [ ]:
# ============================================================
# SETUP — Install dependencies and clone repo if on Colab
# ============================================================
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('Smart_EV_Charging'):
        !git clone https://github.com/x1linwang/Smart_EV_Charging.git
    os.chdir('Smart_EV_Charging')

!pip install -r requirements.txt
print('Setup complete.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Project modules — PROVIDED infrastructure. Do not modify the .py files.
from config import Config
from data_utils import (
    generate_synthetic_prices, load_realistic_prices,
    get_daily_price_curve, generate_ev_schedules,
    train_test_split_prices, plot_price_profile, plot_ev_schedules,
)

cfg = Config()
np.random.seed(cfg.seed)
print(cfg.summary())

## 1. Data Generation

### Understanding the Data

This project uses **two types of data**, both generated programmatically (no external CSV files needed):

**1. Electricity Price Data** (`generate_synthetic_prices()`)
- **What**: Hourly wholesale electricity prices in $/MWh, modeled after California ISO (CAISO) day-ahead market patterns.
- **How many**: 365 days x 24 hours = 8,760 hourly price observations by default.
- **Key patterns**: Prices follow a strong daily cycle driven by supply and demand fundamentals:
  - **Overnight valley** (midnight–6 AM): Low demand, abundant baseload generation → ~$15-25/MWh
  - **Solar duck curve dip** (10 AM–3 PM): Solar panels flood the grid with cheap energy → ~$10-20/MWh
  - **Evening peak** (5–9 PM): Solar drops off while demand remains high → ~$50-100/MWh
- **Variation sources**: Seasonal adjustment (summer higher), weekday/weekend effects, random noise, occasional price spikes (1% chance) and negative prices from renewable surplus (0.5% chance at midday).
- **Why synthetic?** Reproducible across students, no API keys needed, works offline in Colab.

**2. EV Fleet Schedules** (`generate_ev_schedules()`)
- **Arrivals**: Uniformly distributed between 2 PM and 8 PM.
- **Departures**: Uniformly distributed between 5 AM and 9 AM the next morning.
- **Arrival SoC**: Truncated normal distribution (mean=35%, std=10%, clipped to [20%, 60%]).
- **Feasibility guarantee**: Each EV is guaranteed enough time to physically reach 90% SoC.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Fleet size | 20 EVs | Large enough that transformer coupling matters |
| Battery capacity | 60 kWh | Typical commercial van (Ford E-Transit class) |
| Max charge/discharge rate | ±11 kW | Level 2 AC, bidirectional for V2G |
| Transformer limit | 150 kW | 20×11=220 kW demand vs. 150 kW supply → coordination needed |
| Target SoC | ≥90% at departure | Hard constraint — penalty for missed deadlines |
| Time resolution | 15 minutes | 96 steps per 24-hour simulation day |
| V2G degradation cost | $0.04/kWh | Battery wear from discharge cycling |

In [ ]:
# Generate synthetic electricity prices (CAISO-calibrated)
prices_df = generate_synthetic_prices(cfg)
print(f"Generated {len(prices_df)} hourly price records over {cfg.price.num_days_history} days")
print(f"Price stats: mean=${prices_df['price_mwh'].mean():.1f}, "
      f"min=${prices_df['price_mwh'].min():.1f}, "
      f"max=${prices_df['price_mwh'].max():.1f}/MWh")

# Visualize price patterns
plot_price_profile(prices_df, num_days=10)

In [ ]:
# Generate stochastic EV arrival/departure schedules
schedules = generate_ev_schedules(cfg)
print("Sample EV schedules:")
for ev in schedules[:5]:
    print(f"  {ev}")
print(f"  ... ({len(schedules)} total)")

# Visualize fleet schedules (Gantt chart — color = charging urgency)
plot_ev_schedules(schedules, cfg)

In [ ]:
# Chronological train/test split (no data leakage)
train_df, test_df = train_test_split_prices(prices_df, cfg)

# Extract one day's price curve for simulation (15-min resolution)
price_curve = get_daily_price_curve(prices_df, day_index=0, cfg=cfg)
print(f"Day 0 price curve: {price_curve.shape[0]} steps, "
      f"range ${price_curve.min():.1f}\u2013${price_curve.max():.1f}/MWh")

plt.figure(figsize=(12, 4))
steps = np.arange(len(price_curve))
hours = (cfg.time.start_hour + steps * cfg.time.dt_hours) % 24
plt.plot(hours, price_curve, 'b-', linewidth=1.5)
plt.xlabel('Hour of Day'); plt.ylabel('Price ($/MWh)')
plt.title('Day 0 Price Curve (15-min resolution)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 2. Baseline Strategies (Heuristics)

Before applying any AI, we establish performance baselines with simple **rule-based strategies**:

- **ASAP**: Charge every EV at max power immediately upon arrival. Simple but price-blind.
- **ALAP**: Wait until the last moment, then charge at full power. Risky under transformer constraints.
- **Round Robin**: Sort EVs by urgency, allocate transformer capacity to the most urgent first. Best heuristic but still ignores prices.

All three run on the **same scenario** for fair comparison.

In [ ]:
from heuristics import run_all_heuristics, run_heuristic
from environment import make_env, plot_episode_results

heuristic_results = run_all_heuristics(
    cfg, schedules=schedules, price_curve=price_curve, verbose=True
)

In [ ]:
# Visualize the Round Robin strategy in detail
env_rr = make_env(cfg, schedules=schedules, price_curve=price_curve)
_ = run_heuristic(env_rr, strategy='round_robin')
plot_episode_results(env_rr, title='Round Robin Baseline')

## 3. LP Optimal Solution (Gold Standard Benchmark)

### Why a Linear Program with Perfect Foresight?

The LP optimizer (`optimizer.py`) solves the charging problem to **global optimality** using **complete knowledge of all future prices**. No real-time controller can access this information, so the LP serves as a **theoretical ceiling** — the absolute best anyone could possibly do.

The gap between LP cost and any online method represents the **\"cost of uncertainty\"** — the economic penalty of not having perfect foresight. This gap quantifies how much a fleet operator should be willing to pay for a perfect price forecast.

### LP Formulation

**Minimize:** Total electricity cost + V2G degradation cost

**Subject to:**
- **Power bounds:** 0 ≤ charge ≤ max_charge_kw when connected; -max_discharge_kw ≤ discharge ≤ 0 for V2G
- **SoC dynamics:** SoC[t+1] = SoC[t] + (power[t] × dt × efficiency) / capacity
- **SoC bounds:** soc_min ≤ SoC[t] ≤ soc_max at every step
- **Departure target:** SoC[departure] ≥ 90% (hard constraint)
- **Transformer limit:** sum(charging_power[t]) ≤ 150 kW at every step (coupling constraint)

Solved by `scipy.optimize.linprog` with the HiGHS solver.

In [ ]:
from optimizer import solve_optimal_schedule, plot_optimal_schedule

lp_result = solve_optimal_schedule(
    schedules, price_curve, cfg, allow_v2g=True, verbose=True
)
plot_optimal_schedule(lp_result, price_curve, cfg)

## 4. Pillar 1: Machine Learning — Price Forecasting

### What the Provided ML Pipeline Does

The baseline in `ml_forecaster.py` uses **XGBoost** and **Random Forest** to predict the **next hour's electricity price** from engineered tabular features:

- **Lag features**: price at t-1, t-2, t-3, t-6, t-12, t-24, t-168 (same hour last week)
- **Rolling statistics**: mean/std over 6h, 24h, 168h windows
- **Cyclical time encodings**: sin/cos of hour, day-of-week, month
- **Price momentum**: 1h and 24h price differences
- **Peak indicators**: is_peak, is_solar, is_overnight flags

**Target:** `price[t+1]` (next hour). All features use only past/current data — no lookahead bias.

### TODO: Improve the Feature Engineering

Add new features to improve forecasting accuracy:
1. Additional lags (t-48, t-72)
2. Interaction features (hour × is_weekend)
3. Higher-frequency Fourier terms
4. Exponentially weighted moving averages
5. Hyperparameter tuning

In [ ]:
from ml_forecaster import (
    engineer_features, train_ml_models, evaluate_ml_models,
    get_feature_importance, plot_ml_results, plot_feature_importance,
    save_ml_model,
)

# ============================================================
# TODO: Write your own custom feature engineering function.
# Start by copying the baseline and adding new features.
# ============================================================
#
# def engineer_features_custom(df, cfg):
#     feat = engineer_features(df, cfg)  # Start with baseline
#     # --- ADD YOUR NEW FEATURES ---
#     # feat['price_lag_48'] = feat['price_mwh'].shift(48)
#     # feat['hour_x_weekend'] = feat['hour'] * feat['is_weekend']
#     # feat['hour_sin2'] = np.sin(4 * np.pi * feat['hour'] / 24)
#     # feat['price_ewm_12'] = feat['price_mwh'].ewm(span=12).mean()
#     # feat.dropna(inplace=True)
#     # feat.reset_index(drop=True, inplace=True)
#     # return feat

# Use baseline (replace with your custom function)
train_feat = engineer_features(train_df, cfg)
test_feat = engineer_features(test_df, cfg)
feature_cols = [c for c in train_feat.columns if c not in {'datetime','price_mwh','target','day_index'}]
print(f'Feature count: {len(feature_cols)}')
print(f'Train: {len(train_feat)} samples, Test: {len(test_feat)} samples')

In [ ]:
# Train and evaluate ML models
ml_models = train_ml_models(train_feat, cfg)
ml_results = evaluate_ml_models(ml_models, test_feat)
plot_ml_results(ml_results, test_feat, num_days=7)

In [ ]:
# Feature importance analysis
imp_df = get_feature_importance(ml_models, top_n=15)
print(imp_df.to_string(index=False))
plot_feature_importance(imp_df)

In [ ]:
# SAVE YOUR ML MODEL FOR SUBMISSION
save_ml_model(ml_models, path='submission/ml_model.pkl')

## 5. Pillar 2: Deep Learning — LSTM Sequence Forecasting

### How the LSTM Works

Unlike XGBoost (which needs hand-crafted features), the LSTM learns representations directly from **raw price sequences**.

**Input:** A sliding window of **168 hours** (1 week) of prices, shaped `(batch, 168, 1)`.

```
[p_{t-167}, p_{t-166}, ..., p_t]  \u2192  predict: p_{t+1}
```

**Architecture:** 2-layer LSTM (hidden=64, dropout=0.2) \u2192 Linear(64\u219232) \u2192 ReLU \u2192 Linear(32\u21921)

**Training:** Adam + MSE loss, ReduceLROnPlateau, gradient clipping, early stopping.

**Normalization:** Z-score using training set statistics (saved with model for inference).

### TODO: Improve the LSTM

1. Vary hidden_size (32, 64, 128, 256)
2. Try GRU instead of LSTM
3. Add attention mechanism
4. Multi-feature input (price + hour + day_of_week)
5. Different sequence lengths (48h, 336h)

In [ ]:
from dl_forecaster import train_lstm, evaluate_lstm, plot_dl_results, save_lstm_model, PriceLSTM

train_prices = train_df['price_mwh'].values
test_prices = test_df['price_mwh'].values

# ============================================================
# TODO (optional): Define a custom LSTM model class.
# Must accept (input_size, hidden_size, num_layers, dropout)
# and return shape (batch,) from forward().
# ============================================================
# import torch.nn as nn
# class CustomPriceLSTM(nn.Module):
#     def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
#         super().__init__()
#         self.hidden_size = hidden_size
#         self.num_layers = num_layers
#         # YOUR ARCHITECTURE HERE
#     def forward(self, x):
#         pass  # Return (batch,)

In [ ]:
# Train LSTM — adjust hyperparameters as needed
# cfg.forecast.lstm_hidden_size = 128
# cfg.forecast.lstm_num_layers = 3
cfg.forecast.lstm_epochs = 30

lstm_dict = train_lstm(
    train_prices,
    val_prices=test_prices[:2000],
    cfg=cfg,
    verbose=True,
    # model_class=CustomPriceLSTM,  # Uncomment to use your custom model
)

In [ ]:
# Evaluate LSTM and compare with ML
lstm_results = evaluate_lstm(lstm_dict, test_prices)
plot_dl_results(lstm_results, ml_results=ml_results, history=lstm_dict['history'])

print('\n' + '='*55)
print('ML vs. DL Comparison')
print('='*55)
print(f'{"Model":<15} {"MAE ($/MWh)":<15} {"RMSE ($/MWh)":<15} {"R\u00b2":<10}')
print('-'*55)
for name, res in ml_results.items():
    print(f'{name:<15} ${res["mae"]:<14.2f} ${res["rmse"]:<14.2f} {res["r2"]:<10.4f}')
print(f'{"LSTM":<15} ${lstm_results["mae"]:<14.2f} ${lstm_results["rmse"]:<14.2f} {lstm_results["r2"]:<10.4f}')

In [ ]:
# SAVE YOUR LSTM MODEL FOR SUBMISSION
save_lstm_model(lstm_dict, path='submission/lstm_model.pth')

## 6. Pillar 3: Reinforcement Learning — PPO Agent

### How the RL Environment Works

The EV charging problem is formulated as a **Markov Decision Process** in `environment.py`:

**State space (63-dim):** Per-EV (SoC, time_to_departure, is_connected) \u00d7 20 EVs + global (time, price, load_fraction).

**CRITICAL:** The agent sees the **current** price but **NOT future prices**. It must learn to anticipate price patterns from the time-of-day signal alone.

**Action space (20-dim, continuous [-1, +1]):** +1 = charge at 11 kW, 0 = idle, -1 = discharge at 11 kW (V2G). Transformer constraint enforced by proportional scaling.

**Reward (baseline):** `-(price_weight \u00d7 step_cost) - deadline_penalty - overload_penalty`
- `step_cost = charging_cost - v2g_revenue + degradation_cost`
- `deadline_penalty = 100 \u00d7 shortfall` (fires when EV departs below 90% SoC)
- `overload_penalty = 50 \u00d7 overload_fraction`

**PPO** (actor-critic, MLP [256\u2192128]) trains on randomized scenarios for ~100k-200k steps (~5-10 min).

### TODO: Design a Custom Reward Function

1. Adjust penalty magnitudes
2. Add smoothness reward (penalize power swings)
3. Explicitly reward V2G revenue
4. Time-aware shaping (partial credit for progress toward target)
5. Lexicographic priorities (targets first, then cost)

In [ ]:
from rl_agent import train_rl_agent, evaluate_rl_agent, run_single_episode, plot_training_curves

# ============================================================
# TODO: Define your custom reward function.
# ============================================================
# def my_custom_reward(step_cost, deadline_penalty, overload_penalty,
#                      charging_energy, discharging_energy,
#                      soc_array, cfg, current_step, price):
#     reward = -1.5 * step_cost - 200.0 * deadline_penalty - overload_penalty
#     return reward

cfg.rl.total_timesteps = 100_000  # Increase to 200_000+ for better policies

rl_result = train_rl_agent(
    cfg,
    verbose=True,
    # custom_reward_fn=my_custom_reward,
)

In [ ]:
# Training progress
plot_training_curves(rl_result['training_stats'])

In [ ]:
# Evaluate on the same scenario as heuristics/LP
rl_eval = evaluate_rl_agent(
    rl_result['model'], cfg,
    schedules=schedules, price_curve=price_curve, n_episodes=5,
)

rl_episode = run_single_episode(
    rl_result['model'], cfg,
    schedules=schedules, price_curve=price_curve,
)
plot_episode_results(rl_episode['env'], title='PPO RL Agent')

In [ ]:
# SAVE YOUR RL AGENT FOR SUBMISSION
import os
os.makedirs('submission', exist_ok=True)
rl_result['model'].save('submission/rl_agent')
print('RL agent saved to submission/rl_agent.zip')

## 7. Comprehensive Strategy Comparison

Consolidates all results onto a single dashboard. Key metrics:
- **Net cost ($):** Lower is better.
- **EVs meeting target:** Higher is better (20/20 = perfect).
- **V2G revenue ($):** Money earned selling energy back during peak hours.

In [ ]:
comparison = []
for h in heuristic_results:
    comparison.append(h)

comparison.append({
    'strategy': 'lp_optimal',
    'net_cost': lp_result['net_cost'],
    'total_cost': lp_result['total_cost'],
    'v2g_revenue': lp_result['v2g_revenue'],
    'degradation_cost': lp_result.get('degradation_cost', 0),
    'penalties': 0,
    'evs_meeting_target': lp_result['evs_meeting_target'],
    'total_evs': lp_result['total_evs'],
})

comparison.append(rl_episode)

comp_df = pd.DataFrame(comparison)
display_cols = ['strategy','net_cost','v2g_revenue','penalties','evs_meeting_target','total_evs']
existing_cols = [c for c in display_cols if c in comp_df.columns]
print('\n' + '='*80)
print('STRATEGY COMPARISON DASHBOARD')
print('='*80)
print(comp_df[existing_cols].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
strategies = comp_df['strategy'].values
costs = comp_df['net_cost'].values
colors = ['#94A3B8','#94A3B8','#94A3B8','#0D9488','#F59E0B'][:len(strategies)]

ax1 = axes[0]
bars = ax1.bar(strategies, costs, color=colors)
ax1.set_ylabel('Net Cost ($)'); ax1.set_title('Net Electricity Cost')
ax1.tick_params(axis='x', rotation=45)
for bar, cost in zip(bars, costs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'${cost:.1f}', ha='center', va='bottom', fontsize=9)
ax1.grid(True, axis='y', alpha=0.3)

ax2 = axes[1]
targets = comp_df['evs_meeting_target'].values
total = comp_df['total_evs'].values[0]
ax2.bar(strategies, targets, color=colors)
ax2.axhline(y=total, color='red', linestyle='--', label=f'Total ({total})')
ax2.set_ylabel('EVs Meeting Target'); ax2.set_title('Deadline Compliance')
ax2.tick_params(axis='x', rotation=45); ax2.legend(); ax2.grid(True, axis='y', alpha=0.3)

ax3 = axes[2]
if 'v2g_revenue' in comp_df.columns:
    ax3.bar(strategies, comp_df['v2g_revenue'].values, color=colors)
    ax3.set_ylabel('V2G Revenue ($)'); ax3.set_title('V2G Revenue')
    ax3.tick_params(axis='x', rotation=45); ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 8. Pillar 4: Agentic AI — LLM Orchestrator

The agentic AI module demonstrates how an LLM (GPT-4o) can orchestrate all AI modules via **tool-calling**.

- **Mock mode (default):** Works without an API key. Use this in Colab.
- **Live mode:** Set `OPENAI_API_KEY` for real GPT-4o responses.

In [ ]:
from agentic_ai import create_agent

best_ml_model = ml_models['models'].get('xgboost', None)
agent = create_agent(
    cfg=cfg, schedules=schedules, price_curve=price_curve,
    ml_model=best_ml_model,
    lstm_dict=lstm_dict if 'lstm_dict' in dir() else None,
    rl_model=rl_result['model'] if 'rl_result' in dir() else None,
    prices_df=prices_df,
)

print(agent.chat('What is the cheapest charging plan for tonight?'))
print('\n' + '='*60 + '\n')
print(agent.chat('Compare all strategies and tell me which is best.'))

## 9. TODO: Analysis Report (Student Deliverable)

Write a detailed analysis answering the questions below. Support your answers with **specific numbers** from your experiments.

**1. ML vs. DL Forecasting**
- Which model achieved lower MAE? By how much?
- Top 5 most important XGBoost features — why do they rank highest?
- Does LSTM beat XGBoost here? When might DL outperform ML for price forecasting?
- What custom features/architecture changes did you try, and what was the effect?

**2. RL vs. LP Optimal: Cost of Uncertainty**
- Compute: `(RL_cost - LP_cost) / LP_cost \u00d7 100%`
- Did the RL agent learn to shift charging to cheap hours? Use V2G during peaks?
- Describe your custom reward function design and its effect on the learned policy.
- Main failure modes of the RL agent?

**3. Overall Strategy Ranking**
- Rank all strategies by net cost. Which would you deploy at a real depot? Why?
- What is the value of the agentic AI layer for production use?

**4. Extensions (optional, for extra credit)**

In [ ]:
# YOUR ANALYSIS CODE AND DISCUSSION HERE
# Add as many cells as needed for plots, tables, and written analysis.

## 10. Submission

Your submission folder should contain:
```
submission/
- ml_model.pkl       # From Section 4
- lstm_model.pth     # From Section 5
- rl_agent.zip       # From Section 6
- student_info.json  # Generated below
- main.ipynb         # This notebook with your analysis
```

The CA will run an autograder that evaluates your models on a **hidden holdout dataset** and produces a ranked leaderboard.

In [ ]:
import json, os

student_name = 'YOUR NAME HERE'   # <-- Replace
student_uni  = 'YOUR_UNI_HERE'    # <-- Replace

os.makedirs('submission', exist_ok=True)
with open('submission/student_info.json', 'w') as f:
    json.dump({'name': student_name, 'uni': student_uni}, f, indent=2)

print(f'Student info saved: {student_name} ({student_uni})')
print('\nSubmission checklist:')
for fname in ['ml_model.pkl', 'lstm_model.pth', 'rl_agent.zip', 'student_info.json']:
    path = os.path.join('submission', fname)
    exists = os.path.exists(path)
    size = f'({os.path.getsize(path)/1024:.1f} KB)' if exists else ''
    print(f'  [{"OK" if exists else "MISSING"}] {fname} {size}')